In [1]:
# =====================================================================
# onset_V2_ext4_biggerCRNN_3seeds
# Bigger CRNN (CNN 32-64-128 + BiGRU 3 layers / 128 hidden), NO augmentation
# Trained 3 times with 3 seeds -> report mean ± std, save weights of each run
# Previous single run: onset F1 0.847 | offset 0.835 | male 0.816 | female 0.890
# =====================================================================

import torch, torchaudio
import torch.nn as nn
import torch.optim as optim
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import random
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAMPLE_RATE = 44100
HOP_SECONDS = 0.010
HOP   = int(HOP_SECONDS * SAMPLE_RATE)
N_FFT = 2048
N_MELS = 128
LABELS = ["Rest", "Sound", "Onset", "Offset"]
label_to_id = {n: i for i, n in enumerate(LABELS)}

ROOT = Path("/kaggle/input/datasets/alirahmatpour/vocalset-onset")
ANNOT_CSV_DIR = ROOT / "annot_ext4_upload" / "annot_ext4_upload" / "annot"
AUDIO_DIR     = ROOT / "kaggle_upload" / "kaggle_upload" / "audio"
OUT_DIR       = Path("/kaggle/working")          # weights are saved here

COLS = ["Sequence","Start","End","Duration","Type","AvgF0","MedF0","MinF0","MaxF0",
        "StdF0","AvgF0_inSTD","EstMIDI","GTNote","GTFreq","GTMIDI","Lyric",
        "GTDurName","GTDur","IntPrev","IntNext","Extra"]

def read_annot(csv_path):
    a = pd.read_csv(csv_path, skiprows=17, header=None, names=COLS)
    return a[["Start","End","Type"]].dropna(subset=["Start","End","Type"])

def frame_labels(annot, n_frames):
    labs = np.full(n_frames, label_to_id["Rest"], dtype=np.int64)
    for _, r in annot.iterrows():
        if r["Type"] == "Sound":
            lo, hi = int(r["Start"]/HOP_SECONDS), int(r["End"]/HOP_SECONDS)
            labs[lo:hi] = label_to_id["Sound"]
    for _, r in annot[annot["Type"]=="Sound"].iterrows():
        off = int(r["End"]/HOP_SECONDS)
        if off < n_frames: labs[off] = label_to_id["Offset"]
    for _, r in annot[annot["Type"]=="Sound"].iterrows():
        on = int(r["Start"]/HOP_SECONDS)
        if on < n_frames: labs[on] = label_to_id["Onset"]
    return labs

csv_lookup = {p.stem: p for p in ANNOT_CSV_DIR.rglob("*.csv")}
pairs, dropped = [], 0
for wav in sorted(AUDIO_DIR.rglob("*.wav")):      # sorted -> same order every run
    if wav.stem not in csv_lookup:
        continue
    wf2, sr2 = torchaudio.load(str(wav))
    dur = wf2.shape[1] / sr2
    ann = read_annot(csv_lookup[wav.stem])
    if not len(ann) or abs(ann["End"].max()/dur - 1.0) > 0.1:
        dropped += 1
        continue
    singer = wav.parts[wav.parts.index("audio") + 1]
    sex = "female" if singer.startswith("female") else "male"
    pairs.append({"name": wav.stem, "wav": wav, "csv": csv_lookup[wav.stem],
                  "singer": singer, "sex": sex})

print("device      :", device, "| GPU:", torch.cuda.is_available())
print("clean files :", len(pairs), "| dropped:", dropped, "(expect ~2537 / ~147)")

device      : cuda | GPU: True
clean files : 2537 | dropped: 147 (expect ~2537 / ~147)


In [2]:
random.seed(42)

males   = sorted({p["singer"] for p in pairs if p["sex"]=="male"})
females = sorted({p["singer"] for p in pairs if p["sex"]=="female"})
random.shuffle(males); random.shuffle(females)

test_singers  = set(males[:1]  + females[:1])
val_singers   = set(males[1:3] + females[1:3])
train_singers = set(males[3:]  + females[3:])

def bucket(p):
    if p["singer"] in test_singers: return "test"
    if p["singer"] in val_singers:  return "val"
    return "train"

train_items = [p for p in pairs if bucket(p)=="train"]
val_items   = [p for p in pairs if bucket(p)=="val"]
test_items  = [p for p in pairs if bucket(p)=="test"]

print("train:", len(train_items), "| val:", len(val_items), "| test:", len(test_items))
print("test singers:", sorted(test_singers), "(expect female4, male6)")

train: 1719 | val: 566 | test: 252
test singers: ['female4', 'male6'] (expect female4, male6)


In [3]:
SOFT_WIDTH = 2

def soft_targets(annot, n_frames):
    onset_t  = np.zeros(n_frames, dtype=np.float32)
    offset_t = np.zeros(n_frames, dtype=np.float32)
    def bump(arr, center):
        for d in range(-SOFT_WIDTH, SOFT_WIDTH+1):
            j = center + d
            if 0 <= j < n_frames:
                arr[j] = max(arr[j], np.exp(-(d*d)/(2*(SOFT_WIDTH/2)**2)))
    for _, r in annot[annot["Type"]=="Sound"].iterrows():
        bump(onset_t,  int(r["Start"]/HOP_SECONDS))
        bump(offset_t, int(r["End"]/HOP_SECONDS))
    return onset_t, offset_t

def state_from_frames(y_hard):
    return np.where(y_hard == label_to_id["Rest"], 0, 1).astype(np.int64)

class SeqDataset(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p = self.items[i]
        wf, sr = torchaudio.load(str(p["wav"]))
        wave = wf[0]
        nfr  = 1 + (wave.shape[0]//HOP)
        ann  = read_annot(p["csv"])
        hard = frame_labels(ann, nfr)
        state = torch.tensor(state_from_frames(hard))
        on, off = soft_targets(ann, nfr)
        return (wave, state, torch.tensor(on), torch.tensor(off),
                torch.tensor(hard), nfr, p["sex"])

def collate(batch):
    waves, states, ons, offs, hards, lens, sexes = zip(*batch)
    X    = pad_sequence(list(waves),  batch_first=True).unsqueeze(1)
    S    = pad_sequence(list(states), batch_first=True, padding_value=-100)
    On   = pad_sequence(list(ons),    batch_first=True, padding_value=0.0)
    Off  = pad_sequence(list(offs),   batch_first=True, padding_value=0.0)
    Hard = pad_sequence(list(hards),  batch_first=True, padding_value=-100)
    maxF = S.shape[1]
    mask = torch.zeros(len(batch), maxF, dtype=torch.bool)
    for i, L in enumerate(lens): mask[i, :L] = True
    return X, S, On, Off, Hard, mask, torch.tensor(lens), list(sexes)

BATCH = 8
# val/test loaders never shuffle, so they can be built once
val_loader  = DataLoader(SeqDataset(val_items),  batch_size=BATCH, shuffle=False, collate_fn=collate, num_workers=2)
test_loader = DataLoader(SeqDataset(test_items), batch_size=BATCH, shuffle=False, collate_fn=collate, num_workers=2)

def make_train_loader(seed):
    g = torch.Generator(); g.manual_seed(seed)     # shuffle order depends on the seed
    return DataLoader(SeqDataset(train_items), batch_size=BATCH, shuffle=True,
                      collate_fn=collate, num_workers=2, generator=g)

In [4]:
class OnsetCRNN(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.mel   = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
        self.to_db = torchaudio.transforms.AmplitudeToDB()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(32),
            nn.MaxPool2d((2,1)),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(64),
            nn.MaxPool2d((2,1)),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(128),
        )
        feat = 128 * (N_MELS // 4)
        self.rnn = nn.GRU(feat, 128, num_layers=3, batch_first=True,
                          bidirectional=True, dropout=dropout)
        self.drop = nn.Dropout(dropout)
        self.state_head  = nn.Linear(256, 2)
        self.onset_head  = nn.Linear(256, 1)
        self.offset_head = nn.Linear(256, 1)

    def forward(self, x):
        s = self.to_db(self.mel(x))
        s = self.cnn(s)
        B, C, M, T = s.shape
        s = s.reshape(B, C*M, T).transpose(1, 2)
        s, _ = self.rnn(s)
        s = self.drop(s)
        state  = self.state_head(s)
        onset  = self.onset_head(s).squeeze(-1)
        offset = self.offset_head(s).squeeze(-1)
        return state, onset, offset

In [5]:
random.seed(42)
sample_items = random.sample(train_items, min(300, len(train_items)))
sc = np.zeros(2, dtype=np.float64)
for p in sample_items:
    wf_, sr_ = torchaudio.load(str(p["wav"]))
    nfr = 1 + (wf_.shape[1]//HOP)
    s = state_from_frames(frame_labels(read_annot(p["csv"]), nfr))
    sc[0] += (s==0).sum(); sc[1] += (s==1).sum()
state_w = torch.tensor(sc.sum()/(2*np.maximum(sc,1)), dtype=torch.float32).to(device)

state_loss_fn = nn.CrossEntropyLoss(weight=state_w, ignore_index=-100)
soft_loss_fn  = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(20.0).to(device))

def combined_loss(st, on, off, S, On, Off, mask):
    F = min(st.shape[1], S.shape[1])
    ls = state_loss_fn(st[:, :F, :].reshape(-1, 2), S[:, :F].reshape(-1))
    m  = mask[:, :F]
    lo = soft_loss_fn(on[:, :F][m],  On[:, :F][m])
    lf = soft_loss_fn(off[:, :F][m], Off[:, :F][m])
    return ls + lo + lf

print("state class weights:", state_w.cpu().numpy().round(3))

state class weights: [3.734 0.577]


In [6]:
def predict_probs(model, loader):
    model.eval()
    On, Off, T, S = [], [], [], []
    with torch.no_grad():
        for X, St, Onb, Offb, Hard, mask, lens, sexes in loader:
            st, on, off = model(X.to(device))
            on  = torch.sigmoid(on).cpu(); off = torch.sigmoid(off).cpu()
            for b in range(X.size(0)):
                L = lens[b].item()
                On.append(on[b, :L].numpy()); Off.append(off[b, :L].numpy())
                T.append(Hard[b, :L].numpy()); S.append(sexes[b])
    return On, Off, T, S

def pick_peaks(prob, thr, min_gap=3):
    peaks = []
    for i in range(len(prob)):
        if prob[i] < thr: continue
        lo, hi = max(0, i-1), min(len(prob), i+2)
        if prob[i] == prob[lo:hi].max() and (not peaks or i-peaks[-1] >= min_gap):
            peaks.append(i)
    return set(peaks)

def event_f1(prob_list, trues, class_id, thr, tol=2):
    tp = fp = fn = 0
    for prob, t in zip(prob_list, trues):
        pred = pick_peaks(prob, thr)
        t_idx = list(np.where(t==class_id)[0]); matched = set()
        for ti in t_idx:
            hit = next((pi for pi in pred if abs(pi-ti) <= tol and pi not in matched), None)
            if hit is not None: tp += 1; matched.add(hit)
            else: fn += 1
        fp += len(pred - matched)
    prec = tp/(tp+fp) if tp+fp else 0.0
    rec  = tp/(tp+fn) if tp+fn else 0.0
    f1   = 2*prec*rec/(prec+rec) if prec+rec else 0.0
    return prec, rec, f1

def evaluate(model):
    vOn, _, vT, _ = predict_probs(model, val_loader)
    best_thr, best_f1 = 0.5, 0.0
    for thr in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
        _, _, f1 = event_f1(vOn, vT, label_to_id["Onset"], thr)
        if f1 > best_f1: best_f1, best_thr = f1, thr
    On, Off, T, S = predict_probs(model, test_loader)
    res = {"thr": best_thr}
    res["onset"]  = event_f1(On,  T, label_to_id["Onset"],  best_thr)[2]
    res["offset"] = event_f1(Off, T, label_to_id["Offset"], best_thr)[2]
    for sex in ["male", "female"]:
        idx = [i for i in range(len(On)) if S[i]==sex]
        res[sex] = event_f1([On[i] for i in idx], [T[i] for i in idx],
                            label_to_id["Onset"], best_thr)[2]
    return res

In [7]:
EPOCHS = 20
PATIENCE = 4
MAX_FRAMES = 1500

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def train_one(seed):
    set_seed(seed)
    model = OnsetCRNN(dropout=0.3).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    train_loader = make_train_loader(seed)
    w_path = OUT_DIR / f"biggerCRNN_ext4_seed{seed}.pt"

    def run_epoch(loader, train=True):
        model.train() if train else model.eval()
        tot, n = 0.0, 0
        with torch.set_grad_enabled(train):
            for X, S, On, Off, Hard, mask, lens, sexes in loader:
                ms = MAX_FRAMES * HOP
                X, S = X[:, :, :ms].to(device), S[:, :MAX_FRAMES].to(device)
                On, Off = On[:, :MAX_FRAMES].to(device), Off[:, :MAX_FRAMES].to(device)
                mask = mask[:, :MAX_FRAMES].to(device)
                st, on, off = model(X)
                loss = combined_loss(st, on, off, S, On, Off, mask)
                if train:
                    optimizer.zero_grad(); loss.backward(); optimizer.step()
                tot += loss.item()*X.size(0); n += X.size(0)
                del X, S, On, Off, st, on, off, loss
        torch.cuda.empty_cache()
        return tot/n

    best_val, best_epoch, wait = float("inf"), 0, 0
    for epoch in range(1, EPOCHS+1):
        tr = run_epoch(train_loader, True)
        va = run_epoch(val_loader, False)
        flag = ""
        if va < best_val:
            best_val, best_epoch, wait = va, epoch, 0
            torch.save(model.state_dict(), w_path)
            flag = "  <-- best (saved)"
        else:
            wait += 1
        print(f"[seed {seed}] epoch {epoch:2d} | train {tr:.3f} | val {va:.3f}{flag}")
        if wait >= PATIENCE:
            print(f"[seed {seed}] early stop"); break

    model.load_state_dict(torch.load(w_path))
    res = evaluate(model)
    res["seed"], res["best_epoch"] = seed, best_epoch
    print(f"[seed {seed}] best epoch {best_epoch} | thr {res['thr']} | "
          f"onset {res['onset']:.3f} | offset {res['offset']:.3f} | "
          f"male {res['male']:.3f} | female {res['female']:.3f} | saved: {w_path.name}\n")
    del model; torch.cuda.empty_cache()
    return res

In [8]:
SEEDS = [0, 1, 2]
results = [train_one(s) for s in SEEDS]

df = pd.DataFrame(results)[["seed","best_epoch","thr","onset","offset","male","female"]]
print(df.round(3).to_string(index=False))

print("\n=== bigger CRNN, 3 seeds (mean ± std) ===")
for k in ["onset", "offset", "male", "female"]:
    print(f"{k:7s}: {df[k].mean():.3f} ± {df[k].std(ddof=1):.3f}")

df.to_csv(OUT_DIR / "biggerCRNN_ext4_3seeds_results.csv", index=False)
print("\nweights + results saved in", OUT_DIR)

[seed 0] epoch  1 | train 1.380 | val 1.225  <-- best (saved)
[seed 0] epoch  2 | train 0.987 | val 1.129  <-- best (saved)
[seed 0] epoch  3 | train 0.892 | val 1.055  <-- best (saved)
[seed 0] epoch  4 | train 0.848 | val 1.063
[seed 0] epoch  5 | train 0.808 | val 1.016  <-- best (saved)
[seed 0] epoch  6 | train 0.783 | val 0.935  <-- best (saved)
[seed 0] epoch  7 | train 0.750 | val 0.949
[seed 0] epoch  8 | train 0.732 | val 0.917  <-- best (saved)
[seed 0] epoch  9 | train 0.711 | val 0.910  <-- best (saved)
[seed 0] epoch 10 | train 0.708 | val 0.890  <-- best (saved)
[seed 0] epoch 11 | train 0.697 | val 0.893
[seed 0] epoch 12 | train 0.679 | val 0.889  <-- best (saved)
[seed 0] epoch 13 | train 0.657 | val 0.922
[seed 0] epoch 14 | train 0.658 | val 0.918
[seed 0] epoch 15 | train 0.649 | val 0.896
[seed 0] epoch 16 | train 0.627 | val 0.914
[seed 0] early stop
[seed 0] best epoch 12 | thr 0.8 | onset 0.838 | offset 0.829 | male 0.803 | female 0.883 | saved: biggerCRNN_ext4